# BC reach visualization from DGL .bin (normal + multimesh)

This notebook loads one or multiple graphs from a DGL `.bin`, computes hop distance from BC source nodes (Q/H/WALL/ALL), and exports PNG frames, CSV counts, and an optional GIF.

It works with both normal mesh and multimesh graphs as long as node types are present in node features.

In [ ]:
import csv
from collections import deque
from pathlib import Path

import dgl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np

In [ ]:
def tensor_to_numpy(t):
    return t.detach().cpu().numpy()


def get_static_features(g):
    # Preferred format in this project: g.ndata['static']
    if 'static' in g.ndata:
        static = tensor_to_numpy(g.ndata['static'])
        if static.shape[1] < 4:
            raise ValueError('g.ndata["static"] has <4 columns, cannot read one-hot node types.')
        return static

    # Fallback: if only x exists and starts with one-hot static features
    if 'x' in g.ndata:
        x = tensor_to_numpy(g.ndata['x'])
        if x.shape[1] < 4:
            raise ValueError('g.ndata["x"] has <4 columns, cannot read one-hot node types.')
        return x

    raise ValueError('No usable static node features found (expected ndata["static"] or ndata["x"]).')


def get_bc_masks_from_onehot(onehot, bc_type):
    q_mask = np.all(onehot == np.array([0, 0, 1, 0]), axis=1)
    h_mask = np.all(onehot == np.array([0, 1, 0, 0]), axis=1)
    wall_mask = np.all(onehot == np.array([0, 0, 0, 1]), axis=1)
    all_mask = q_mask | h_mask | wall_mask

    if bc_type == 'q':
        source = q_mask
    elif bc_type == 'h':
        source = h_mask
    elif bc_type == 'wall':
        source = wall_mask
    elif bc_type == 'all':
        source = all_mask
    else:
        raise ValueError(f'Unsupported bc_type={bc_type}')

    return source, q_mask, h_mask, wall_mask


def build_adjacency_from_edges(num_nodes, src, dst):
    adj = [set() for _ in range(num_nodes)]
    for s, d in zip(src, dst):
        s = int(s)
        d = int(d)
        adj[s].add(d)
        adj[d].add(s)
    return [list(v) for v in adj]


def multi_source_hop_distance(adjacency, source_mask):
    n = len(adjacency)
    dist = np.full(n, -1, dtype=np.int32)
    q = deque()

    for node in np.where(source_mask)[0]:
        dist[node] = 0
        q.append(int(node))

    while q:
        u = q.popleft()
        for v in adjacency[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)

    return dist


def reconstruct_xy_from_edge_rel(g, edge_feat_key='x'):
    # Reconstruct coordinates from edge relation: pos_i - pos_j = [xrel, yrel]
    if edge_feat_key not in g.edata:
        raise ValueError(f'Edge feature key {edge_feat_key!r} not found in g.edata.')

    edge_feat = tensor_to_numpy(g.edata[edge_feat_key])
    if edge_feat.shape[1] < 2:
        raise ValueError('Edge features must have at least 2 columns [xrel, yrel].')

    src_t, dst_t = g.edges()
    src = tensor_to_numpy(src_t).astype(np.int64)
    dst = tensor_to_numpy(dst_t).astype(np.int64)
    rel = edge_feat[:, :2].astype(np.float64)

    n = g.num_nodes()
    coords = np.full((n, 2), np.nan, dtype=np.float64)

    outgoing = [[] for _ in range(n)]
    for e in range(src.shape[0]):
        outgoing[src[e]].append((dst[e], rel[e]))

    visited = np.zeros(n, dtype=bool)
    x_offset = 0.0
    component_gap = 5.0

    for root in range(n):
        if visited[root]:
            continue

        queue = deque([root])
        visited[root] = True
        coords[root] = np.array([x_offset, 0.0], dtype=np.float64)

        component_nodes = [root]
        while queue:
            u = queue.popleft()
            pu = coords[u]
            for v, rel_uv in outgoing[u]:
                candidate = pu - rel_uv
                if not visited[v]:
                    visited[v] = True
                    coords[v] = candidate
                    queue.append(v)
                    component_nodes.append(v)

        comp_xy = coords[component_nodes]
        comp_min = np.min(comp_xy[:, 0])
        comp_max = np.max(comp_xy[:, 0])
        x_offset += (comp_max - comp_min) + component_gap

    coords = coords.astype(np.float32)
    return coords[:, 0], coords[:, 1]


def get_xy_for_plot(g):
    # If graph already contains node coordinates, use them directly
    for key in ['pos', 'xy', 'coord', 'coords', 'mesh_pos']:
        if key in g.ndata:
            arr = tensor_to_numpy(g.ndata[key])
            if arr.ndim == 2 and arr.shape[1] >= 2:
                return arr[:, 0].astype(np.float32), arr[:, 1].astype(np.float32), f'ndata[{key}]'

    # Otherwise reconstruct from edge relative vectors
    x, y = reconstruct_xy_from_edge_rel(g, edge_feat_key='x')
    return x, y, 'reconstructed_from_edata[x][:,0:2]'


def save_counts_csv(rows, output_csv):
    fieldnames = [
        'step',
        'reached_including_bc',
        'reached_excluding_bc',
        'newly_reached_including_bc',
        'newly_reached_excluding_bc',
        'fraction_excluding_bc',
    ]
    with output_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_gif_with_pillow(frame_paths, gif_path, fps):
    try:
        from PIL import Image
    except ImportError:
        print('Pillow not installed: GIF not created (PNG frames are available).')
        return False

    if not frame_paths:
        return False

    duration_ms = max(1, int(round(1000.0 / max(fps, 1e-6))))
    frames = [Image.open(p) for p in frame_paths]
    try:
        frames[0].save(
            gif_path,
            save_all=True,
            append_images=frames[1:],
            duration=duration_ms,
            loop=0,
        )
    finally:
        for fr in frames:
            fr.close()

    return True

In [ ]:
# -----------------------------
# User parameters
# -----------------------------
BIN_FILE = '/path/to/mesh_or_multimesh_base.bin'

# You can process multiple graphs from the same .bin
# Example if graph 0=normal and graph 1=multimesh:
GRAPH_SPECS = [
    ('normal', 0),
    ('multimesh', 1),
]

BC_TYPE = 'q'      # one of: 'q', 'h', 'wall', 'all'
MAX_HOPS = 10

OUTPUT_DIR = Path('./gnn_modulus_test/jupyter/bc_message_passing_viz_from_bin')
DPI = 180
NODE_SIZE = 3.0
FPS = 1.2
DRAW_EDGES = True
MAX_EDGES_TO_DRAW = 300000
SKIP_GIF = False

In [ ]:
bin_path = Path(BIN_FILE).expanduser().resolve()
graphs, _ = dgl.load_graphs(str(bin_path))

print('Loaded bin:', bin_path)
print('Number of graphs:', len(graphs))
for i, g in enumerate(graphs):
    print(f'  graph[{i}] -> nodes={g.num_nodes():,}, edges={g.num_edges():,}, ndata={list(g.ndata.keys())}, edata={list(g.edata.keys())}')

In [ ]:
def run_viz_for_graph(g, graph_name, graph_index):
    out_dir = OUTPUT_DIR / f'{graph_name}_idx{graph_index}'
    frames_dir = out_dir / 'frames'
    frames_dir.mkdir(parents=True, exist_ok=True)

    static = get_static_features(g)
    onehot = static[:, :4]
    source_mask, q_mask, h_mask, wall_mask = get_bc_masks_from_onehot(onehot, BC_TYPE)

    source_count = int(source_mask.sum())
    if source_count == 0:
        raise RuntimeError(
            f'No BC source nodes for graph={graph_name} index={graph_index}, BC_TYPE={BC_TYPE}. '
            f'Counts: q={int(q_mask.sum())}, h={int(h_mask.sum())}, wall={int(wall_mask.sum())}'
        )

    src_t, dst_t = g.edges()
    src = tensor_to_numpy(src_t).astype(np.int64)
    dst = tensor_to_numpy(dst_t).astype(np.int64)

    adjacency = build_adjacency_from_edges(g.num_nodes(), src, dst)
    dist = multi_source_hop_distance(adjacency, source_mask)

    x, y, xy_source = get_xy_for_plot(g)

    non_bc_count = int((~source_mask).sum())
    max_hops = max(1, int(MAX_HOPS))

    # Precompute undirected edge segments for plotting
    edge_segments = None
    if DRAW_EDGES:
        undirected_mask = src < dst
        us = src[undirected_mask]
        ud = dst[undirected_mask]

        if us.shape[0] > MAX_EDGES_TO_DRAW:
            pick = np.linspace(0, us.shape[0] - 1, MAX_EDGES_TO_DRAW, dtype=np.int64)
            us = us[pick]
            ud = ud[pick]

        edge_segments = np.stack([np.column_stack([x[us], y[us]]), np.column_stack([x[ud], y[ud]])], axis=1)

    csv_rows = []
    frame_paths = []

    for step in range(1, max_hops + 1):
        reached = (dist >= 0) & (dist <= step)
        reached_no_bc = reached & (~source_mask)
        frontier = (dist == step)
        frontier_no_bc = frontier & (~source_mask)
        unreached_or_far = (~reached) & (~source_mask)

        reached_including_bc = int(reached.sum())
        reached_excluding_bc = int(reached_no_bc.sum())
        newly_reached_including_bc = int(frontier.sum())
        newly_reached_excluding_bc = int(frontier_no_bc.sum())
        frac_excl = reached_excluding_bc / non_bc_count if non_bc_count > 0 else float('nan')

        csv_rows.append({
            'step': step,
            'reached_including_bc': reached_including_bc,
            'reached_excluding_bc': reached_excluding_bc,
            'newly_reached_including_bc': newly_reached_including_bc,
            'newly_reached_excluding_bc': newly_reached_excluding_bc,
            'fraction_excluding_bc': frac_excl,
        })

        fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

        if edge_segments is not None:
            lc = LineCollection(edge_segments, colors='#e6e6e6', linewidths=0.2, alpha=0.8, zorder=1)
            ax.add_collection(lc)

        if np.any(unreached_or_far):
            ax.scatter(x[unreached_or_far], y[unreached_or_far], s=NODE_SIZE, c='#d8d8d8', alpha=0.8, linewidths=0, zorder=2)
        if np.any(reached_no_bc):
            ax.scatter(x[reached_no_bc], y[reached_no_bc], s=NODE_SIZE, c='#1f77b4', alpha=0.9, linewidths=0, zorder=3)
        if np.any(frontier_no_bc):
            ax.scatter(x[frontier_no_bc], y[frontier_no_bc], s=max(4.0, NODE_SIZE * 2.2), c='#ff7f0e', alpha=1.0, linewidths=0, zorder=4)
        if np.any(source_mask):
            ax.scatter(x[source_mask], y[source_mask], s=max(5.0, NODE_SIZE * 2.4), c='#d62728', alpha=1.0, linewidths=0, zorder=5)

        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(
            f'{graph_name} (idx={graph_index}) | BC={BC_TYPE.upper()} | step {step}/{max_hops}\n'
            f'Reached (excl. BC): {reached_excluding_bc:,}/{non_bc_count:,} ({100.0 * frac_excl:.2f}%) | '
            f'Frontier: {newly_reached_excluding_bc:,} | XY source: {xy_source}',
            fontsize=11,
        )

        frame_path = frames_dir / f'bc_reach_step_{step:02d}.png'
        fig.savefig(frame_path, dpi=DPI)
        plt.close(fig)
        frame_paths.append(frame_path)

    counts_csv = out_dir / 'bc_reach_counts.csv'
    save_counts_csv(csv_rows, counts_csv)

    gif_path = out_dir / 'bc_reach_1_to_k.gif'
    gif_ok = False
    if not SKIP_GIF:
        gif_ok = save_gif_with_pillow(frame_paths, gif_path, fps=FPS)

    print('---')
    print(f'Graph: {graph_name} (idx={graph_index})')
    print(f'Nodes={g.num_nodes():,}, Edges={g.num_edges():,}')
    print(f'Source BC nodes={source_count:,}, Non-BC nodes={non_bc_count:,}')
    print(f'Frames: {frames_dir}')
    print(f'CSV: {counts_csv}')
    if SKIP_GIF:
        print('GIF: skipped (SKIP_GIF=True)')
    elif gif_ok:
        print(f'GIF: {gif_path}')
    else:
        print('GIF: not created (install Pillow).')

    return {
        'name': graph_name,
        'index': graph_index,
        'out_dir': out_dir,
        'gif_path': gif_path if gif_ok else None,
        'counts_csv': counts_csv,
    }


results = []
for graph_name, graph_index in GRAPH_SPECS:
    if graph_index < 0 or graph_index >= len(graphs):
        raise IndexError(f'graph_index={graph_index} out of range for {len(graphs)} graph(s).')
    results.append(run_viz_for_graph(graphs[graph_index], graph_name, graph_index))

results